# 0022 — Evidence가 있는 SQL Publish Candidate만 제공

SQL Analytics Endpoint의 Candidate View와 Batch의 `candidate_count`에 동일 Batch의 Evidence 존재 조건을 적용한다.

In [ ]:
sql_endpoint_server = None
sql_endpoint_database = None
applied_by_run_id = None

In [ ]:
import hashlib
import json
import struct
from datetime import datetime, timezone

import pyodbc
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, TimestampType

MIGRATION_VERSION = 22
MIGRATION_NAME = "filter-sql-publish-candidates-with-evidence"
MIGRATION_TABLE = "gold._schema_migrations"
EXPECTED_PRIOR_VERSION = 21
EXPECTED_PRIOR_NAME = "filter-publish-rereview-candidate-by-cve"
TOKEN_SCOPE = "https://database.windows.net/"
SQL_COPT_SS_ACCESS_TOKEN = 1256

CANDIDATE_EVIDENCE_EXISTS = """
EXISTS (
    SELECT 1
    FROM gold.gold_rereview_candidate_evidence AS evidence
    WHERE evidence.batch_id = candidate.batch_id
      AND evidence.candidate_id = candidate.candidate_id
)
""".strip()

VIEW_DDLS = {
    "gold.publish_rereview_batch": f"""
CREATE OR ALTER VIEW gold.publish_rereview_batch AS
SELECT batch.batch_id, batch.run_id AS ds_run_id, batch.status, batch.as_of,
       batch.generated_at, batch.policy_version, batch.schema_version,
       CAST((
           SELECT COUNT_BIG(*)
           FROM gold.gold_rereview_candidate AS candidate
           WHERE candidate.batch_id = batch.batch_id
             AND candidate.cve_id IS NOT NULL
             AND {CANDIDATE_EVIDENCE_EXISTS}
       ) AS bigint) AS candidate_count
FROM gold.gold_rereview_batch AS batch
WHERE batch.status = 'completed'
""",
    "gold.publish_rereview_candidate": f"""
CREATE OR ALTER VIEW gold.publish_rereview_candidate AS
SELECT candidate.batch_id, candidate.candidate_id,
       CASE WHEN candidate.cve_id IS NOT NULL THEN 'CVE' ELSE 'GHSA' END AS subject_type,
       COALESCE(candidate.cve_id, candidate.ghsa_id) AS subject_id,
       candidate.ghsa_id, candidate.cve_id, candidate.priority,
       candidate.primary_trigger_type AS change_type,
       candidate.reason_summary, candidate.selected_at
FROM gold.gold_rereview_candidate AS candidate
JOIN gold.gold_rereview_batch AS batch
  ON batch.batch_id = candidate.batch_id
WHERE batch.status = 'completed'
  AND candidate.cve_id IS NOT NULL
  AND {CANDIDATE_EVIDENCE_EXISTS}
""",
}

EXPECTED_COLUMNS = {
    "publish_rereview_batch": ["batch_id", "ds_run_id", "status", "as_of", "generated_at", "policy_version", "schema_version", "candidate_count"],
    "publish_rereview_candidate": ["batch_id", "candidate_id", "subject_type", "subject_id", "ghsa_id", "cve_id", "priority", "change_type", "reason_summary", "selected_at"],
}
SPEC = {
    "view_ddls": VIEW_DDLS,
    "expected_columns": EXPECTED_COLUMNS,
    "candidate_filter": "completed CVE candidate with evidence in the same batch",
}
CHECKSUM = hashlib.sha256(
    json.dumps(SPEC, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()

if not sql_endpoint_server or not sql_endpoint_database:
    raise ValueError("sql_endpoint_server and sql_endpoint_database parameters are required")
if not spark.catalog.tableExists(MIGRATION_TABLE):
    raise RuntimeError("migration history missing")

history_rows = (
    spark.table(MIGRATION_TABLE)
    .select(F.col("version").cast("int").alias("version"), "name", "checksum")
    .collect()
)
history = {int(row["version"]): row for row in history_rows}
if len(history) != len(history_rows):
    raise RuntimeError("duplicate migration history version")
if (
    EXPECTED_PRIOR_VERSION not in history
    or history[EXPECTED_PRIOR_VERSION]["name"] != EXPECTED_PRIOR_NAME
):
    raise RuntimeError("migration 0021 must be applied first")
if any(version > MIGRATION_VERSION for version in history):
    raise RuntimeError("database is ahead of migration 0022")
already_applied = MIGRATION_VERSION in history
if already_applied and (
    history[MIGRATION_VERSION]["name"] != MIGRATION_NAME
    or history[MIGRATION_VERSION]["checksum"] != CHECKSUM
):
    raise RuntimeError("applied migration 0022 differs from definition")

token = notebookutils.credentials.getToken(TOKEN_SCOPE).encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token)}s", len(token), token)
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER=tcp:{sql_endpoint_server},1433;DATABASE={sql_endpoint_database};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)
connection = pyodbc.connect(
    connection_string,
    attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct},
    autocommit=True,
)
try:
    cursor = connection.cursor()
    for ddl in VIEW_DDLS.values():
        cursor.execute(ddl)

    for view_name, expected in EXPECTED_COLUMNS.items():
        cursor.execute(
            "SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS "
            "WHERE TABLE_SCHEMA = 'gold' AND TABLE_NAME = ? "
            "ORDER BY ORDINAL_POSITION",
            (view_name,),
        )
        actual = [row[0] for row in cursor.fetchall()]
        if actual != expected:
            raise RuntimeError(
                f"SQL view column mismatch: {view_name}: actual={actual}, expected={expected}"
            )

    checks = {
        "non_completed_batches": "SELECT COUNT(*) FROM gold.publish_rereview_batch WHERE status <> 'completed' OR status IS NULL",
        "invalid_candidate_identifiers": "SELECT COUNT(*) FROM gold.publish_rereview_candidate WHERE cve_id IS NULL OR subject_type <> 'CVE' OR subject_id <> cve_id",
        "duplicate_candidates": "SELECT COUNT(*) FROM (SELECT batch_id, candidate_id FROM gold.publish_rereview_candidate GROUP BY batch_id, candidate_id HAVING COUNT(*) > 1) duplicated",
        "candidate_without_evidence": "SELECT COUNT(*) FROM gold.publish_rereview_candidate AS candidate WHERE NOT EXISTS (SELECT 1 FROM gold.publish_rereview_candidate_evidence AS evidence WHERE evidence.batch_id = candidate.batch_id AND evidence.candidate_id = candidate.candidate_id)",
        "candidate_count_mismatch": "SELECT COUNT(*) FROM gold.publish_rereview_batch AS batch OUTER APPLY (SELECT COUNT_BIG(*) AS actual_count FROM gold.publish_rereview_candidate AS candidate WHERE candidate.batch_id = batch.batch_id) AS counts WHERE batch.candidate_count <> counts.actual_count",
    }
    validation = {}
    for name, query in checks.items():
        cursor.execute(query)
        validation[name] = cursor.fetchone()[0]
    failed = {name: count for name, count in validation.items() if count != 0}
    if failed:
        raise RuntimeError(f"SQL publish validation failed: {failed}")

    cursor.execute("SELECT COUNT(*) FROM gold.publish_rereview_candidate")
    candidate_count = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM gold.publish_rereview_candidate_evidence")
    evidence_count = cursor.fetchone()[0]
    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.gold_rereview_candidate AS candidate
        JOIN gold.gold_rereview_batch AS batch ON batch.batch_id = candidate.batch_id
        WHERE batch.status = 'completed'
          AND candidate.cve_id IS NOT NULL
          AND NOT EXISTS (
              SELECT 1
              FROM gold.gold_rereview_candidate_evidence AS evidence
              WHERE evidence.batch_id = candidate.batch_id
                AND evidence.candidate_id = candidate.candidate_id
          )
    """)
    excluded_without_evidence_count = cursor.fetchone()[0]
finally:
    connection.close()

if not already_applied:
    migration_schema = StructType([
        StructField("version", IntegerType(), False),
        StructField("name", StringType(), False),
        StructField("applied_at", TimestampType(), False),
        StructField("applied_by_run_id", StringType(), True),
        StructField("checksum", StringType(), False),
    ])
    row = [(
        MIGRATION_VERSION,
        MIGRATION_NAME,
        datetime.now(timezone.utc).replace(tzinfo=None),
        applied_by_run_id,
        CHECKSUM,
    )]
    spark.createDataFrame(row, migration_schema).write.mode("append").saveAsTable(MIGRATION_TABLE)

run_result = {
    "status": "skipped" if already_applied else "ok",
    "migration_version": MIGRATION_VERSION,
    "migration_name": MIGRATION_NAME,
    "checksum": CHECKSUM,
    "candidate_count": candidate_count,
    "evidence_count": evidence_count,
    "excluded_without_evidence_count": excluded_without_evidence_count,
    "validation": validation,
}
print(json.dumps(run_result, sort_keys=True))
notebookutils.notebook.exit(json.dumps(run_result, sort_keys=True))